# Preprocessing and Feature Engineering

## Clean the Kaggle data

Keep only the hybrid era (2014 onward) and the drivers on the 2026 grid (`data/recent/grid_2026.csv`). The output has one row per driver per Grand Prix, with a `podium` label (1 = finished in the top 3).

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
RECENT = Path("../data/recent")
PROCESSED = Path("../data/processed")
HYBRID_START = 2014


def read_raw(name):
    # The Kaggle CSVs use \N for missing values
    return pd.read_csv(RAW / f"{name}.csv", na_values="\\N")


races = read_raw("races")
results = read_raw("results")
qualifying = read_raw("qualifying")
drivers = read_raw("drivers")
constructors = read_raw("constructors")
status = read_raw("status")
grid_2026 = pd.read_csv(RECENT / "grid_2026.csv")

### Match the 2026 grid to Kaggle driver IDs

Drivers are matched by full name, with accents removed (Hülkenberg, Pérez). Three-letter codes aren't unique across eras (Vergne also used VER), so they aren't used for matching.

In [2]:
def name_key(first, last):
    text = f"{first} {last}"
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().lower().strip()


drivers["name_key"] = [name_key(f, l) for f, l in zip(drivers.forename, drivers.surname)]
grid_2026["name_key"] = [name_key(f, l) for f, l in zip(grid_2026.first_name, grid_2026.last_name)]

grid_matched = grid_2026.merge(drivers[["driverId", "name_key"]], on="name_key", how="left")
unmatched = grid_matched[grid_matched.driverId.isna()]
print(f"Matched {grid_matched.driverId.notna().sum()} of {len(grid_matched)} drivers")
print("No Kaggle history (debuted 2025 or later):", ", ".join(unmatched.first_name + " " + unmatched.last_name))

current_driver_ids = grid_matched.driverId.dropna().astype(int)

Matched 18 of 22 drivers
No Kaggle history (debuted 2025 or later): Gabriel Bortoleto, Andrea Kimi Antonelli, Arvid Lindblad, Isack Hadjar


### Filter and join

In [3]:
hybrid_races = races.loc[races.year >= HYBRID_START, ["raceId", "year", "round", "circuitId", "name", "date"]]
hybrid_races = hybrid_races.rename(columns={"name": "race_name"})

quali_pos = qualifying[["raceId", "driverId", "position"]].rename(columns={"position": "quali_position"})

df = (
    results[results.driverId.isin(current_driver_ids)]
    .merge(hybrid_races, on="raceId", how="inner")
    .merge(drivers[["driverId", "code", "forename", "surname"]], on="driverId", how="left")
    .merge(constructors[["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId", how="left")
    .merge(status, on="statusId", how="left")
    .merge(quali_pos, on=["raceId", "driverId"], how="left")
)

df["date"] = pd.to_datetime(df["date"])
df["podium"] = (df["positionOrder"] <= 3).astype(int)

df = df[[
    "raceId", "year", "round", "date", "race_name", "circuitId",
    "driverId", "code", "forename", "surname", "constructorId", "constructor",
    "quali_position", "grid", "position", "positionOrder", "points", "status", "podium",
]].sort_values(["date", "positionOrder"]).reset_index(drop=True)

df.head()

,raceId,year,round,date,race_name,circuitId,driverId,code,forename,surname,constructorId,constructor,quali_position,grid,position,positionOrder,points,status,podium
0,900,2014,1,2014-03-16,Australian Grand Prix,1,4,ALO,Fernando,Alonso,6,Ferrari,5.0,5,4.0,4,12.0,Finished,0
1,900,2014,1,2014-03-16,Australian Grand Prix,1,822,BOT,Valtteri,Bottas,3,Williams,10.0,15,5.0,5,10.0,Finished,0
2,900,2014,1,2014-03-16,Australian Grand Prix,1,807,HUL,Nico,Hülkenberg,10,Force India,7.0,7,6.0,6,8.0,Finished,0
3,900,2014,1,2014-03-16,Australian Grand Prix,1,815,PER,Sergio,Pérez,10,Force India,16.0,16,10.0,10,1.0,Finished,0
4,900,2014,1,2014-03-16,Australian Grand Prix,1,1,HAM,Lewis,Hamilton,131,Mercedes,1.0,1,NaN,19,0.0,Engine,0


## Add 2025 and 2026 from FastF1

Kaggle stops at 2024, so newer seasons come from `data/recent/results_2025_2026.csv`, made by `python -m src.fetch_recent`. FastF1 doesn't use Kaggle's IDs, so the code below maps drivers by name, teams by name, and circuits by location. Teams and drivers that are new since 2024 (Audi, Cadillac, the four rookies) get new IDs above Kaggle's range.

In [4]:
recent = pd.read_csv(RECENT / "results_2025_2026.csv", parse_dates=["date"])
circuits = read_raw("circuits")

# Drivers: Kaggle driverId by name; rookies get new IDs
recent["name_key"] = [name_key(f, l) for f, l in zip(recent.first_name, recent.last_name)]
recent = recent[recent.name_key.isin(grid_2026.name_key)]
driver_ids = dict(zip(drivers.name_key, drivers.driverId))
next_id = drivers.driverId.max() + 1
for key in sorted(set(recent.name_key) - set(driver_ids)):
    driver_ids[key] = next_id
    next_id += 1
recent["driverId"] = recent.name_key.map(driver_ids)

# Teams: FastF1 name -> Kaggle constructor name (new teams keep their FastF1 name)
TEAM_TO_KAGGLE = {
    "Red Bull Racing": "Red Bull",
    "Kick Sauber": "Sauber",
    "Alpine": "Alpine F1 Team",
    "Haas F1 Team": "Haas F1 Team",
    "Racing Bulls": "RB F1 Team",
    "RB F1 Team": "RB F1 Team",
}
recent["constructor"] = recent.team.replace(TEAM_TO_KAGGLE)
constructor_ids = dict(zip(constructors.name, constructors.constructorId))
next_id = constructors.constructorId.max() + 1
for name in sorted(set(recent.constructor) - set(constructor_ids)):
    constructor_ids[name] = next_id
    next_id += 1
recent["constructorId"] = recent.constructor.map(constructor_ids)

# Circuits: FastF1 location -> Kaggle location (Madrid is new and gets a new ID)
LOCATION_TO_KAGGLE = {
    "Barcelona": "Montmeló",
    "Montréal": "Montreal",
    "Monte Carlo": "Monte-Carlo",
    "Monaco": "Monte-Carlo",
    "Spa-Francorchamps": "Spa",
    "Miami Gardens": "Miami",
    "Lusail": "Al Daayen",
    "Yas Island": "Abu Dhabi",
    "Yas Marina": "Abu Dhabi",
}
circuit_ids = dict(zip(circuits.location, circuits.circuitId))
next_id = circuits.circuitId.max() + 1
recent_locations = recent.location.replace(LOCATION_TO_KAGGLE)
for loc in sorted(set(recent_locations) - set(circuit_ids)):
    print("New circuit:", loc)
    circuit_ids[loc] = next_id
    next_id += 1
recent["circuitId"] = recent_locations.map(circuit_ids)

# Races: new raceIds continue after Kaggle's
race_keys = recent[["year", "round"]].drop_duplicates().sort_values(["year", "round"])
race_keys["raceId"] = range(races.raceId.max() + 1, races.raceId.max() + 1 + len(race_keys))
recent = recent.merge(race_keys, on=["year", "round"])

recent = recent.rename(columns={"first_name": "forename", "last_name": "surname"})
recent["podium"] = (recent["positionOrder"] <= 3).astype(int)
recent = recent[df.columns]

print(f"FastF1 rows: {len(recent):,} from {recent.raceId.nunique()} races")
print("Teams:", ", ".join(sorted(recent.constructor.unique())))

FastF1 rows: 884 from 44 races
Teams: Alpine F1 Team, Aston Martin, Audi, Cadillac F1 Team, Ferrari, Haas F1 Team, McLaren, Mercedes, RB F1 Team, Red Bull, Sauber, Williams


In [5]:
df = pd.concat([df, recent], ignore_index=True).sort_values(["date", "positionOrder"]).reset_index(drop=True)
df.groupby("year").agg(races=("raceId", "nunique"), rows=("raceId", "size"), podium_rate=("podium", "mean"))

,races,rows,podium_rate
year,,,
2014,19,95,0.263158
2015,19,132,0.151515
2016,21,155,0.174194
2017,20,184,0.168478
2018,21,231,0.160173
2019,21,252,0.210317
2020,17,203,0.236453
2021,22,264,0.242424
2022,22,287,0.229965


### Quick checks

In [6]:
print(f"Rows: {len(df):,}  |  Races: {df.raceId.nunique()}  |  Seasons: {df.year.min()}-{df.year.max()}")
print(f"Podium rate: {df.podium.mean():.1%}")
print("Missing values:")
print(df.isna().sum()[lambda s: s > 0])

Rows: 3,398  |  Races: 272  |  Seasons: 2014-2026
Podium rate: 18.9%
Missing values:
quali_position     11
position          471
dtype: int64


In [7]:
df.groupby("code").agg(starts=("raceId", "count"), podiums=("podium", "sum"), first_season=("year", "min")) \
  .sort_values("podiums", ascending=False)

,starts,podiums,first_season
code,,,
HAM,271,155,2014
VER,253,139,2015
BOT,248,67,2014
LEC,193,56,2018
NOR,172,50,2019
PER,246,36,2014
RUS,172,32,2019
SAI,252,29,2015
PIA,90,28,2023


In [8]:
PROCESSED.mkdir(exist_ok=True)
df.to_csv(PROCESSED / "results_hybrid_2026_drivers.csv", index=False)